In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.hospital_env import HospitalEnv

data = pd.read_csv("../data/synthetic_hospital_data.csv")

# -----------------------
# Discretization
# -----------------------
arrival_bins = np.linspace(0, 480, 6)
slot_bins = np.linspace(0, 480, 6)
no_show_bins = np.linspace(0, 1, 6)
icu_bins = np.linspace(0, 1, 6)

def discretize_state(state):
    arrival, slot, priority, no_show, icu = state

    a = np.digitize(arrival, arrival_bins)
    s = np.digitize(slot, slot_bins)
    n = np.digitize(no_show, no_show_bins)
    i = np.digitize(icu, icu_bins)

    return (a, s, int(priority), n, i)

# -----------------------
# Q-Learning Agent
# -----------------------
class QLearningAgent:
    def __init__(self, n_actions, lr=0.1, gamma=0.95, epsilon=0.1):
        self.n_actions = n_actions
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.q = {}

    def select_action(self, state):
        if state not in self.q:
            self.q[state] = np.zeros(self.n_actions)

        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.q[state])

    def update(self, state, action, reward, next_state):
        if state not in self.q:
            self.q[state] = np.zeros(self.n_actions)

        target = reward
        if next_state is not None:
            if next_state not in self.q:
                self.q[next_state] = np.zeros(self.n_actions)
            target += self.gamma * np.max(self.q[next_state])

        self.q[state][action] += self.lr * (target - self.q[state][action])

# -----------------------
# Training
# -----------------------
agent = QLearningAgent(n_actions=2)

N = 500
episode_rewards = []
avg_waiting_times = []
resource_utilization_list = []

for episode in range(N):

    env = HospitalEnv(data)   # IMPORTANT: new env each episode
    state = discretize_state(env.reset())
    done = False
    total_reward = 0

    while not done:
        action = agent.select_action(state)

        next_state_raw, reward, done, info = env.step(action)

        if not done:
            next_state = discretize_state(next_state_raw)
        else:
            next_state = None

        agent.update(state, action, reward, next_state)
        state = next_state
        total_reward += reward

    episode_rewards.append(total_reward)

    avg_wait, utilization = env.get_metrics()
    avg_waiting_times.append(avg_wait)
    resource_utilization_list.append(utilization)

    if episode % 50 == 0:
        print(f"Episode {episode}, Reward = {total_reward}")

# -----------------------
# Save Plots
# -----------------------
os.makedirs("../results", exist_ok=True)

plt.figure()
plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Tabular Q-Learning Training Reward")
plt.savefig("../results/tabular_reward_curve.png")
plt.close()

plt.figure()
plt.plot(resource_utilization_list)
plt.xlabel("Episode")
plt.ylabel("Resource Utilization (%)")
plt.title("Resource Utilization vs Episodes")
plt.savefig("../results/tabular_resource_utilization.png")
plt.close()

Episode 0, Reward = -1177
Episode 50, Reward = 45
Episode 100, Reward = -137
Episode 150, Reward = 64
Episode 200, Reward = -230
Episode 250, Reward = -130
Episode 300, Reward = 77
Episode 350, Reward = -152
Episode 400, Reward = -50
Episode 450, Reward = -90
